# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

The rule, in one breath: "If a page is actually being seen — 1,000+ impressions in the last 90 days — the further its CTR sits below 0.2%, the higher it goes in the fix queue. Ties break toward the bigger page, because fixing that one is worth more."

That's the whole thing. One signal, one line, one tiebreak. I deliberately did not score freshness, word count, or raw position depth as drivers — the signal audit showed those don't predict decline. Position and staleness still appear in the reason codes as context for whoever opens the ticket, but they don't move the score.

Why CTR is the driver: the audit's flag-linked test found low-CTR pages decline about 12 points more than high-CTR pages, even after controlling for position, and the effect got stronger the lower CTR went. That was the one signal that survived every check — so the baseline is built on it.

Reason codes it can output:
- ctr_gap_XXpt — the main reason, and how many points CTR sits below the 0.2% line.
- pos_past1 — ranks worse than page 1 (context: hard to get clicks even with a great snippet).
- stale90 — not updated in 90+ days (context for how the fix gets done, not why it scored).
- below_floor — too few impressions to judge; sits at the bottom, nobody acts on it.

In [10]:
import pandas as pd
import numpy as np

df = pd.read_csv(r'C:\Internship\flyrank-ai\flyrank-ml-internship\data\raw\content_refresh_anonymized.csv')
print(f"Rows: {len(df):,} | Columns: {df.shape[1]} | Clients: {df['client_id'].nunique()}")

# ---- the pasted Section 1 cell, verbatim ----
df["needs_attention"] = df["trend_direction"].isin(["down", "flat"]).astype(int)

visible = df["impressions_90d"] >= 1000
ctr_gap = np.clip(0.2 - df["ctr"], 0, None) * 100.0
size_tie = np.log1p(df["impressions_90d"]) / 1000.0

df["baseline_score"] = np.where(visible, ctr_gap + size_tie, 0.0)

pos_past1 = df["avg_position"] > 10
stale90 = df["days_since_last_update"] >= 90

def reason(row):
    if not row["visible"]:
        return "below_floor"
    parts = [f"ctr_gap_{row['ctr_gap_pts']:.0f}pt"]
    if row["pos_past1"]:
        parts.append("pos_past1")
    if row["stale90"]:
        parts.append("stale90")
    return "+".join(parts)

df["visible"] = visible
df["ctr_gap_pts"] = ctr_gap.round(0)
df["pos_past1"] = pos_past1
df["stale90"] = stale90
df["reason_codes"] = df.apply(reason, axis=1)

print(f"{visible.sum():,} pages above the floor, "
      f"{(visible & (df['ctr'] < 0.2)).sum():,} with a weak-CTR reason")

Rows: 30,000 | Columns: 44 | Clients: 32
13,512 pages above the floor, 7,127 with a weak-CTR reason


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

Rank everything by score, highest first, and write the queue out. Everything below the impression floor gets a score of 0 and sinks to the bottom with a below_floor reason — they stay in the file for completeness, but no editor should touch them.

In [11]:
import pandas as pd
import numpy as np
import os

df = pd.read_csv(r'C:\Internship\flyrank-ai\flyrank-ml-internship\data\raw\content_refresh_anonymized.csv')

# --- Section 1 ---
df["needs_attention"] = df["trend_direction"].isin(["down", "flat"]).astype(int)
df["is_declining"] = (df["trend_direction"] == "down").astype(int)

visible = df["impressions_90d"] >= 1000
ctr_gap = np.clip(0.2 - df["ctr"], 0, None) * 100.0
size_tie = np.log1p(df["impressions_90d"]) / 1000.0
df["baseline_score"] = np.where(visible, ctr_gap + size_tie, 0.0)

pos_past1 = df["avg_position"] > 10
stale90 = df["days_since_last_update"] >= 90

def reason(row):
    if not row["visible"]:
        return "below_floor"
    parts = [f"ctr_gap_{row['ctr_gap_pts']:.0f}pt"]
    if row["pos_past1"]:
        parts.append("pos_past1")
    if row["stale90"]:
        parts.append("stale90")
    return "+".join(parts)

df["visible"] = visible
df["ctr_gap_pts"] = ctr_gap.round(0)
df["pos_past1"] = pos_past1
df["stale90"] = stale90
df["reason_codes"] = df.apply(reason, axis=1)

# --- Section 2 (write to temp; in the notebook use work/outputs/) ---
ranked = df.sort_values("baseline_score", ascending=False)
out_cols = ["content_id", "client_id", "baseline_score", "reason_codes", "trend_direction",
            "impressions_90d", "clicks_90d", "ctr", "avg_position", "days_since_last_update"]
os.makedirs(r'C:\Users\preet\AppData\Local\Temp\opencode\out', exist_ok=True)
ranked[out_cols].to_csv(r'C:\Users\preet\AppData\Local\Temp\opencode\out\baseline_action_score.csv', index=False)

# --- evaluation ---
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return float(np.asarray(labels)[order[:k]].mean())

base_rate = df["needs_attention"].mean()
print(f"base rate (flag everything): {base_rate:.3f}")
for k in (20, 50, 100, 200):
    print(f"  P@{k:<3} = {precision_at_k(df['baseline_score'], df['needs_attention'], k):.3f}")
print(f"  P@20 (down-only labels) = {precision_at_k(df['baseline_score'], df['is_declining'], 20):.3f}")

# --- sanity gradient ---
vv = df[visible].copy()
vv["ctr_decile"] = pd.qcut(vv["ctr"], 10, duplicates="drop")
print(vv.groupby("ctr_decile", observed=True)
        .agg(n=("needs_attention", "count"), declining=("needs_attention", "mean"))
        .round(3).to_string())
print('\nOK - no errors')

base rate (flag everything): 0.580
  P@20  = 0.700
  P@50  = 0.780
  P@100 = 0.750
  P@200 = 0.755
  P@20 (down-only labels) = 0.700
                   n  declining
ctr_decile                     
(-0.001, 0.02]  1403      0.681
(0.02, 0.06]    1352      0.706
(0.06, 0.1]     1613      0.649
(0.1, 0.14]     1248      0.640
(0.14, 0.18]    1218      0.577
(0.18, 0.24]    1422      0.591
(0.24, 0.31]    1247      0.577
(0.31, 0.43]    1364      0.532
(0.43, 0.64]    1317      0.516
(0.64, 5.19]    1328      0.459

OK - no errors


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

Every page in the top 20 has CTR = 0.00, so they're all tied at a 20-point gap — the only thing separating them is the size tiebreak. That already tells me the set is meaningful but the exact order inside the top 20 isn't. Reviewing by hand:

In [12]:
top = df.sort_values("baseline_score", ascending=False).head(20).copy()
top["action"] = np.where(top["ctr"] < 0.05, "Rework title/meta — seen but not clicked", "Monitor")
pd.set_option("display.width", 240)
pd.set_option("display.max_colwidth", 34)
print(top[["content_id", "client_id", "baseline_score", "reason_codes", "trend_direction",
           "impressions_90d", "clicks_90d", "ctr", "avg_position",
           "days_since_last_update", "action"]].round(3).to_string(index=False))

          content_id         client_id  baseline_score                   reason_codes trend_direction  impressions_90d  clicks_90d  ctr  avg_position  days_since_last_update                                   action
content_c8e9d6ab9013 client_19581e27de          20.012           ctr_gap_20pt+stale90            down           208678           0  0.0           9.7                     104 Rework title/meta — seen but not clicked
content_fb4bf6555c79 client_6208ef0f77          20.011 ctr_gap_20pt+pos_past1+stale90            down            84093           3  0.0          45.6                     104 Rework title/meta — seen but not clicked
content_6e28a04c07a8 client_6208ef0f77          20.011 ctr_gap_20pt+pos_past1+stale90            down            41226           2  0.0          34.3                     104 Rework title/meta — seen but not clicked
content_bc18d49d8f6b client_6208ef0f77          20.010 ctr_gap_20pt+pos_past1+stale90          stable            32491           1  0.0     

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

The weak picks are obvious once you stare at the list: 6 of the top 20 are stable or growing (rows 4, 7, 15, 16, 17, 20), and a couple of them are pages that rank well (5.6–7.4) yet got zero clicks. That pattern screams "this isn't a snippet problem" — it's more likely a tracking/attribution split (GSC clicks counted separately from GA4 sessions) or a duplicate URL. Rows 15 and 18 were also updated within the last 8 days, so fixing them again would just churn. And rows 7 and 16 sit past position 50 — a title rewrite won't rescue a page Google has parked on page 6. The deeper lesson: because every top-20 page has CTR = 0.00, the score saturates at 20 points and can't tell "bad snippet" apart from "brand-new page" or "irrelevant query." That's exactly the ambiguity a model with more signals should be able to resolve — which is why this baseline is worth beating, not trusting.

Leakage check: the score only reads impressions_90d, ctr, avg_position, and days_since_last_update, none of which come from trend_direction or trend_pct (the label source). The dataset ships no product flags at all — health_score, needs_ctr_fix, is_quick_win, priority_score, action_type, refresh_tier are all absent by design, so there's nothing to leak in. One honest caveat: ctr is a 90-day total, which contains the same window the label's last-30-days is computed from — that's fine for this starter proxy, but a future-outcome label would need features limited to *_prev30 to avoid window overlap.

In [13]:
top = df.sort_values("baseline_score", ascending=False).head(20)
print(f"not declining/flat in top 20: {(top['needs_attention'] == 0).sum()} of 20")
print(f"  up: {(top['trend_direction'] == 'up').sum()} | stable: {(top['trend_direction'] == 'stable').sum()}")
print(f"zero clicks: {(top['clicks_90d'] == 0).sum()} | ctr==0: {(top['ctr'] == 0).sum()}")
print(f"rank worse than 50: {(top['avg_position'] > 50).sum()}")
print(f"updated within 30 days: {(top['days_since_last_update'] <= 30).sum()}")

# Leakage check 1: does any scoring input derive from the label?
feature_cols = ["impressions_90d", "ctr", "avg_position", "days_since_last_update"]
label_cols = ["trend_direction", "trend_pct"]
print("\nfeatures that are also label-derived:", set(feature_cols) & set(label_cols))

# Leakage check 2: confirm no product flags are present to leak in.
product_flags = ["health_score", "needs_ctr_fix", "is_quick_win",
                 "priority_score", "action_type", "refresh_tier"]
print("product flags present in data:", [c for c in product_flags if c in df.columns])

# Leakage check 3: the score is deterministic and snapshot-only (no future values).
print("uses any future window column?",
      any("last_30" in c for c in feature_cols))

not declining/flat in top 20: 6 of 20
  up: 1 | stable: 5
zero clicks: 12 | ctr==0: 20
rank worse than 50: 2
updated within 30 days: 9

features that are also label-derived: set()
product flags present in data: []
uses any future window column? False


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.